# Child Undernutrition Screening System — Complete Google Colab Pipeline

**Central African Republic MICS6 — Statistical Analysis + Machine Learning + Joblib Export**

This notebook is the reproducible workflow for the dissertation *Child Undernutrition Analysis Using Statistical and Machine Learning Methods: A Case Study of the Central African Republic*.

It contains both major components of the study: statistical analysis and machine-learning prediction of **stunting and underweight**.

Final deployment architecture: **React frontend → Flask API → Joblib (.pkl) models → Render**.

## Step 0 — Install and import packages

In [ ]:
# Core packages are pre-installed on Colab; these two usually aren't.
!pip install -q pyreadstat m2cgen
!pip -q install pyreadstat

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, precision_recall_curve, confusion_matrix)
from xgboost import XGBClassifier

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)
print("Packages ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 3.6 MB/s eta 0:00:00
Packages ready.


## Step 1 — Upload and load the MICS6 child dataset

Upload **either** `Data_View_ch.xlsx` **or** `ch.sav` (the SPSS export). The cell
below detects which one you uploaded and loads it accordingly.

In [ ]:
# ================================================================
# STEP 2 — CONNECT GOOGLE DRIVE
# ================================================================

from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CAR_MICS6"
CH_DIR = os.path.join(
    BASE_DIR,
    "Central African Republic MICS6 SPSS Datasets"
)

CH_PATH = os.path.join(CH_DIR, "ch.sav")

print("=" * 80)
print("GOOGLE DRIVE / CAR MICS6 CHECK")
print("=" * 80)

print("CAR MICS6 folder exists:", os.path.exists(BASE_DIR))
print("CH folder exists:", os.path.exists(CH_DIR))
print("ch.sav exists:", os.path.exists(CH_PATH))
print("CH path:", CH_PATH)

Mounted at /content/drive
GOOGLE DRIVE / CAR MICS6 CHECK
CAR MICS6 folder exists: True
CH folder exists: True
ch.sav exists: True
CH path: /content/drive/MyDrive/CAR_MICS6/Central African Republic MICS6 SPSS Datasets/ch.sav


In [ ]:
# STEP 3 — LOAD CAR MICS6 CH DATASET


import pyreadstat

ch, ch_meta = pyreadstat.read_sav(CH_PATH)

print("=" * 80)
print("CAR MICS6 CHILD DATASET LOADED")
print("=" * 80)

print("Rows:", ch.shape[0])
print("Variables:", ch.shape[1])

display(ch.head())

CAR MICS6 CHILD DATASET LOADED
Rows: 9037
Variables: 452


,HH1,HH2,LN,UF1,UF2,UFINT,UF3,UF4,UF5,UF6,...,windex5,windex10,wscoreu,windex5u,windex10u,wscorer,windex5r,windex10r,PSU,stratum
0,1.0,1.0,6.0,1.0,1.0,94.0,6.0,3.0,94.0,90.0,...,5.0,10.0,2.029471,5.0,10.0,NaN,NaN,NaN,1.0,1.0
1,1.0,4.0,12.0,1.0,4.0,93.0,12.0,5.0,93.0,90.0,...,5.0,10.0,0.353551,4.0,8.0,NaN,NaN,NaN,1.0,1.0
2,1.0,4.0,13.0,1.0,4.0,93.0,13.0,5.0,93.0,90.0,...,5.0,10.0,0.353551,4.0,8.0,NaN,NaN,NaN,1.0,1.0
3,1.0,4.0,14.0,1.0,4.0,93.0,14.0,6.0,93.0,90.0,...,5.0,10.0,0.353551,4.0,8.0,NaN,NaN,NaN,1.0,1.0
4,1.0,4.0,15.0,1.0,4.0,93.0,15.0,5.0,93.0,90.0,...,5.0,10.0,0.353551,4.0,8.0,NaN,NaN,NaN,1.0,1.0


In [ ]:
# ================================================================
# STEP 4 — VERIFY THE 20 CH-ONLY PREDICTORS
# ================================================================

predictors_20 = [
    'CAGE',
    'HL4',
    'CA31',
    'IM2',
    'BD2',
    'cdisability',
    'cinsurance',
    'melevel',
    'caretakerdis',
    'HH6',
    'HH7',
    'windex5',
    'religion',
    'ethnicity',
    'CA1',
    'CA14',
    'CA16',
    'CA17',
    'TN3',
    'EC1'
]

available = [col for col in predictors_20 if col in ch.columns]
missing = [col for col in predictors_20 if col not in ch.columns]

print("=" * 80)
print("20 CH-ONLY PREDICTOR CHECK")
print("=" * 80)

print(f"\nExpected predictors: {len(predictors_20)}")
print(f"Available in CH:    {len(available)}")

print("\n✓ AVAILABLE:")
for col in available:
    print(f"  {col}")

print("\n NOT FOUND:")
for col in missing:
    print(f"  {col}")

20 CH-ONLY PREDICTOR CHECK

Expected predictors: 20
Available in CH:    20

✓ AVAILABLE:
  CAGE
  HL4
  CA31
  IM2
  BD2
  cdisability
  cinsurance
  melevel
  caretakerdis
  HH6
  HH7
  windex5
  religion
  ethnicity
  CA1
  CA14
  CA16
  CA17
  TN3
  EC1

 NOT FOUND:


## Step 2 — Define outcomes and leakage-free predictors

Outcomes follow WHO Child Growth Standards, using the MICS6 quality-flag variables to
exclude biologically implausible records:

- **Stunting**: height-for-age z-score (HAZ) < −2 SD
- **Underweight**: weight-for-age z-score (WAZ) < −2 SD
- *(Wasting: weight-for-height z-score (WHZ) < −2 SD — computed for completeness but
  dropped from the final model; see the markdown note after Step 4.)*

**Target-leakage prevention:** no predictor derived from height, weight, age-based
anthropometry, or the MICS quality flags is included — this is essential, since those
variables define the outcomes themselves.

In [ ]:
# ================================================================
# STEP 5 — CREATE STUNTING AND UNDERWEIGHT OUTCOMES
# ================================================================

print("=" * 80)
print("STEP 5 — CREATING NUTRITION OUTCOMES")
print("=" * 80)

# Create outcomes using MICS/WHO anthropometric indicators
ch['stunting'] = np.nan
ch['underweight'] = np.nan

# Stunting: HAZ < -2, only where HAZ quality flag is valid
valid_haz = ch['HAZFLAG'] == 0
ch.loc[valid_haz, 'stunting'] = (
    ch.loc[valid_haz, 'HAZ'] < -2
).astype(int)

# Underweight: WAZ < -2, only where WAZ quality flag is valid
valid_waz = ch['WAZFLAG'] == 0
ch.loc[valid_waz, 'underweight'] = (
    ch.loc[valid_waz, 'WAZ'] < -2
).astype(int)

print("\nSTUNTING")
print("-" * 50)
print(ch['stunting'].value_counts(dropna=False))
print("Valid cases:", ch['stunting'].notna().sum())
print("Prevalence:", round(ch['stunting'].mean() * 100, 2), "%")

print("\nUNDERWEIGHT")
print("-" * 50)
print(ch['underweight'].value_counts(dropna=False))
print("Valid cases:", ch['underweight'].notna().sum())
print("Prevalence:", round(ch['underweight'].mean() * 100, 2), "%")

STEP 5 — CREATING NUTRITION OUTCOMES

STUNTING
--------------------------------------------------
stunting
0.0    5736
1.0    2865
NaN     436
Name: count, dtype: int64
Valid cases: 8601
Prevalence: 33.31 %

UNDERWEIGHT
--------------------------------------------------
underweight
0.0    6516
1.0    2208
NaN     313
Name: count, dtype: int64
Valid cases: 8724
Prevalence: 25.31 %


In [ ]:
# ================================================================
# STEP 6 — FINAL 20-PREDICTOR MODELING DATASET
# ================================================================

predictors_20 = [
    'CAGE',
    'HL4',
    'CA31',
    'IM2',
    'BD2',
    'cdisability',
    'cinsurance',
    'melevel',
    'caretakerdis',
    'HH6',
    'HH7',
    'windex5',
    'religion',
    'ethnicity',
    'CA1',
    'CA14',
    'CA16',
    'CA17',
    'TN3',
    'EC1'
]

targets = ['stunting', 'underweight']

# Create modeling dataset
model_data = ch[predictors_20 + targets].copy()

print("=" * 80)
print("STEP 6 — FINAL MODELING DATASET")
print("=" * 80)

print("\nShape:", model_data.shape)
print("Predictors:", len(predictors_20))
print("Targets:", targets)

print("\nPredictor columns:")
print(predictors_20)

print("\nMissing values:")
print(model_data.isna().sum().sort_values(ascending=False))

print("\nOutcome distributions:")
for target in targets:
    print(f"\n{target.upper()}")
    print(model_data[target].value_counts(dropna=False))

display(model_data.head())

STEP 6 — FINAL MODELING DATASET

Shape: (9037, 22)
Predictors: 20
Targets: ['stunting', 'underweight']

Predictor columns:
['CAGE', 'HL4', 'CA31', 'IM2', 'BD2', 'cdisability', 'cinsurance', 'melevel', 'caretakerdis', 'HH6', 'HH7', 'windex5', 'religion', 'ethnicity', 'CA1', 'CA14', 'CA16', 'CA17', 'TN3', 'EC1']

Missing values:
TN3             4029
IM2             3821
BD2             3821
CA31            3821
cdisability     3592
stunting         436
underweight      313
CAGE             114
EC1              114
cinsurance       114
CA1              114
CA14             114
CA16             114
CA17             114
HL4                0
ethnicity          0
windex5            0
religion           0
HH6                0
caretakerdis       0
melevel            0
HH7                0
dtype: int64

Outcome distributions:

STUNTING
stunting
0.0    5736
1.0    2865
NaN     436
Name: count, dtype: int64

UNDERWEIGHT
underweight
0.0    6516
1.0    2208
NaN     313
Name: count, dtype: int64


,CAGE,HL4,CA31,IM2,BD2,cdisability,cinsurance,melevel,caretakerdis,HH6,...,religion,ethnicity,CA1,CA14,CA16,CA17,TN3,EC1,stunting,underweight
0,23.0,1.0,6.0,4.0,2.0,NaN,2.0,2.0,2.0,1.0,...,2.0,7.0,1.0,2.0,2.0,2.0,1.0,3.0,1.0,0.0
1,39.0,2.0,NaN,NaN,NaN,1.0,2.0,1.0,2.0,1.0,...,2.0,6.0,2.0,2.0,2.0,2.0,NaN,0.0,0.0,0.0
2,39.0,2.0,NaN,NaN,NaN,1.0,2.0,1.0,2.0,1.0,...,2.0,6.0,2.0,2.0,2.0,2.0,NaN,0.0,0.0,0.0
3,37.0,2.0,NaN,NaN,NaN,1.0,2.0,3.0,2.0,1.0,...,2.0,6.0,2.0,2.0,2.0,2.0,NaN,0.0,0.0,0.0
4,29.0,2.0,2.0,4.0,2.0,1.0,2.0,1.0,2.0,1.0,...,2.0,6.0,2.0,2.0,2.0,2.0,NaN,0.0,0.0,0.0


## Step 2A — Statistical Analysis

This section implements the statistical component of the study before machine-learning modelling.

It includes:
- descriptive distributions and prevalence of stunting and underweight;
- Chi-square tests for associations between categorical predictors and each outcome;
- Cramér's V effect size for categorical associations;
- Mann–Whitney U tests comparing child age between outcome groups.

Statistical significance describes association in the study sample and is not interpreted as causality. Machine-learning feature importance is also interpreted separately from statistical significance.


In [ ]:
# ================================================================
# STEP 7 — DESCRIPTIVE STATISTICAL ANALYSIS
# ================================================================

print("=" * 80)
print("STEP 7 — DESCRIPTIVE STATISTICS")
print("=" * 80)

# ------------------------------------------------
# 1. Numeric variable: Child age
# ------------------------------------------------

print("\n1. CHILD AGE (MONTHS)")
print("-" * 60)

print(model_data['CAGE'].describe())

# ------------------------------------------------
# 2. Outcome prevalence
# ------------------------------------------------

print("\n2. OUTCOME PREVALENCE")
print("-" * 60)

for target in ['stunting', 'underweight']:
    valid = model_data[target].dropna()

    cases = int((valid == 1).sum())
    non_cases = int((valid == 0).sum())
    total = len(valid)
    prevalence = (cases / total) * 100

    print(f"\n{target.upper()}")
    print(f"Valid observations : {total:,}")
    print(f"Non-cases          : {non_cases:,}")
    print(f"Cases              : {cases:,}")
    print(f"Prevalence         : {prevalence:.2f}%")

# ------------------------------------------------
# 3. Categorical predictor distributions
# ------------------------------------------------

categorical_predictors = [
    'HL4',
    'CA31',
    'IM2',
    'BD2',
    'cdisability',
    'cinsurance',
    'melevel',
    'caretakerdis',
    'HH6',
    'HH7',
    'windex5',
    'religion',
    'ethnicity',
    'CA1',
    'CA14',
    'CA16',
    'CA17',
    'TN3',
    'EC1'
]

print("\n3. CATEGORICAL PREDICTOR DISTRIBUTIONS")
print("-" * 60)

for col in categorical_predictors:
    print(f"\n{col}")
    print(model_data[col].value_counts(dropna=False).sort_index())

STEP 7 — DESCRIPTIVE STATISTICS

1. CHILD AGE (MONTHS)
------------------------------------------------------------
count    8923.000000
mean       29.320184
std        17.324268
min         0.000000
25%        14.000000
50%        30.000000
75%        44.000000
max        59.000000
Name: CAGE, dtype: float64

2. OUTCOME PREVALENCE
------------------------------------------------------------

STUNTING
Valid observations : 8,601
Non-cases          : 5,736
Cases              : 2,865
Prevalence         : 33.31%

UNDERWEIGHT
Valid observations : 8,724
Non-cases          : 6,516
Cases              : 2,208
Prevalence         : 25.31%

3. CATEGORICAL PREDICTOR DISTRIBUTIONS
------------------------------------------------------------

HL4
HL4
1.0    4499
2.0    4538
Name: count, dtype: int64

CA31
CA31
1.0      216
2.0     2415
3.0      523
4.0     1318
5.0      146
6.0      446
96.0     111
98.0       8
99.0      33
NaN     3821
Name: count, dtype: int64

IM2
IM2
1.0    1707
2.0     162
3.0 

In [ ]:
# ================================================================
# STEP 8 — RECODE MICS MISSING / DON'T-KNOW CODES
# ================================================================

print("=" * 80)
print("STEP 8 — CLEANING MICS SPECIAL CODES")
print("=" * 80)

analysis_data = model_data.copy()

# MICS special missing / don't know codes
sentinel_codes = {8, 9, 98, 99}

# All categorical predictors
categorical_predictors = [
    'HL4',
    'CA31',
    'IM2',
    'BD2',
    'cdisability',
    'cinsurance',
    'melevel',
    'caretakerdis',
    'HH6',
    'HH7',
    'windex5',
    'religion',
    'ethnicity',
    'CA1',
    'CA14',
    'CA16',
    'CA17',
    'TN3',
    'EC1'
]

# Recode special MICS values to missing
for col in categorical_predictors:
    analysis_data[col] = analysis_data[col].apply(
        lambda x: np.nan if pd.notna(x) and x in sentinel_codes else x
    )

print("\nSpecial MICS codes have been converted to NaN.")

print("\nRemaining missing values:")
missing_summary = (
    analysis_data.isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_summary[missing_summary > 0])

print("\n✓ STEP 8 COMPLETED")

STEP 8 — CLEANING MICS SPECIAL CODES

Special MICS codes have been converted to NaN.

Remaining missing values:
TN3             4029
IM2             3864
CA31            3862
BD2             3850
cdisability     3592
ethnicity       1053
caretakerdis     913
stunting         436
underweight      313
CA1              158
CA17             137
CA14             136
cinsurance       134
CA16             129
EC1              122
CAGE             114
melevel            3
dtype: int64

✓ STEP 8 COMPLETED


In [ ]:
# ================================================================
# STEP 9 — STATISTICAL ASSOCIATION ANALYSIS
# ================================================================

from scipy.stats import chi2_contingency, mannwhitneyu

print("=" * 80)
print("STEP 9 — STATISTICAL ASSOCIATION ANALYSIS")
print("=" * 80)

categorical_predictors = [
    'HL4',
    'CA31',
    'IM2',
    'BD2',
    'cdisability',
    'cinsurance',
    'melevel',
    'caretakerdis',
    'HH6',
    'HH7',
    'windex5',
    'religion',
    'ethnicity',
    'CA1',
    'CA14',
    'CA16',
    'CA17',
    'TN3',
    'EC1'
]

def cramers_v(table):
    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()
    r, k = table.shape

    if n == 0 or min(r - 1, k - 1) == 0:
        return np.nan

    return np.sqrt(
        (chi2 / n) /
        min(k - 1, r - 1)
    )

results = []

for target in ['stunting', 'underweight']:

    print(f"\n{'=' * 80}")
    print(f"OUTCOME: {target.upper()}")
    print(f"{'=' * 80}")

    # ------------------------------------------------------------
    # Categorical predictors — Chi-square
    # ------------------------------------------------------------

    for predictor in categorical_predictors:

        temp = analysis_data[[predictor, target]].dropna()

        if temp[predictor].nunique() < 2 or temp[target].nunique() < 2:
            continue

        table = pd.crosstab(
            temp[predictor],
            temp[target]
        )

        chi2, p, dof, expected = chi2_contingency(table)

        v = cramers_v(table)

        results.append({
            'Target': target,
            'Predictor': predictor,
            'Test': 'Chi-square',
            'N': len(temp),
            'Statistic': chi2,
            'P_Value': p,
            'Effect_Size': v
        })

    # ------------------------------------------------------------
    # Child age — Mann-Whitney U
    # ------------------------------------------------------------

    temp = analysis_data[['CAGE', target]].dropna()

    age_0 = temp.loc[temp[target] == 0, 'CAGE']
    age_1 = temp.loc[temp[target] == 1, 'CAGE']

    if len(age_0) > 0 and len(age_1) > 0:

        statistic, p = mannwhitneyu(
            age_0,
            age_1,
            alternative='two-sided'
        )

        results.append({
            'Target': target,
            'Predictor': 'CAGE',
            'Test': 'Mann-Whitney U',
            'N': len(temp),
            'Statistic': statistic,
            'P_Value': p,
            'Effect_Size': np.nan
        })

# ------------------------------------------------------------
# Results dataframe
# ------------------------------------------------------------

statistical_results_df = pd.DataFrame(results)

# Multiple-testing adjustment: Benjamini-Hochberg FDR
from statsmodels.stats.multitest import multipletests

statistical_results_df['Adjusted_P_Value'] = np.nan

for target in statistical_results_df['Target'].unique():

    mask = statistical_results_df['Target'] == target

    pvals = statistical_results_df.loc[mask, 'P_Value'].values

    if len(pvals) > 0:
        adjusted = multipletests(
            pvals,
            method='fdr_bh'
        )[1]

        statistical_results_df.loc[
            mask,
            'Adjusted_P_Value'
        ] = adjusted

# Significance indicators
statistical_results_df['Significant'] = (
    statistical_results_df['Adjusted_P_Value'] < 0.05
)

# Sort
statistical_results_df = statistical_results_df.sort_values(
    ['Target', 'Adjusted_P_Value']
)

print("\n")
print("=" * 80)
print("STATISTICAL ASSOCIATION RESULTS")
print("=" * 80)

display(statistical_results_df)

STEP 9 — STATISTICAL ASSOCIATION ANALYSIS

OUTCOME: STUNTING

OUTCOME: UNDERWEIGHT


STATISTICAL ASSOCIATION RESULTS


,Target,Predictor,Test,N,Statistic,P_Value,Effect_Size,Adjusted_P_Value,Significant
19,stunting,CAGE,Mann-Whitney U,8601,6.107060e+06,3.456150e-84,NaN,6.912300e-83,True
9,stunting,HH7,Chi-square,8601,2.540220e+02,5.666531e-52,0.171855,5.666531e-51,True
10,stunting,windex5,Chi-square,8601,2.093373e+02,3.689196e-44,0.156009,2.459464e-43,True
6,stunting,melevel,Chi-square,8600,1.989482e+02,7.119183e-43,0.152097,3.559591e-42,True
8,stunting,HH6,Chi-square,8601,1.809230e+02,3.047140e-41,0.145035,1.218856e-40,True
12,stunting,ethnicity,Chi-square,7623,8.583371e+01,8.848597e-16,0.106112,2.949532e-15,True
2,stunting,IM2,Chi-square,5002,3.890211e+01,1.820553e-08,0.088189,5.201581e-08,True
11,stunting,religion,Chi-square,8601,3.290505e+01,3.930381e-06,0.061852,9.825953e-06,True
1,stunting,CA31,Chi-square,5005,2.353391e+01,6.360307e-04,0.068572,1.413401e-03,True
15,stunting,CA16,Chi-square,8586,1.030705e+01,1.325229e-03,0.034647,2.650458e-03,True


## Step 3 — Preprocessing pipeline

Missing values are imputed (median for age, most-frequent category for categoricals)
*inside* a pipeline fitted only on the training fold, so no information leaks from
test to train. Categories are fixed from the **full** dataset up front — this avoids
a real bug we hit during development: if two outcome models are fit on different
subsets of rows, their one-hot encoders can silently learn different column
structures for the same feature.

In [ ]:
# ================================================================
# STEP 10 — PREPARE OUTCOME-SPECIFIC ML DATASETS
# ================================================================

print("=" * 80)
print("STEP 10 — PREPARING ML DATASETS")
print("=" * 80)

stunting_data = analysis_data[
    predictors_20 + ['stunting']
].dropna(subset=['stunting']).copy()

underweight_data = analysis_data[
    predictors_20 + ['underweight']
].dropna(subset=['underweight']).copy()

print("\nSTUNTING DATASET")
print("-" * 60)
print("Shape:", stunting_data.shape)
print("Outcome distribution:")
print(stunting_data['stunting'].value_counts())

print("\nUNDERWEIGHT DATASET")
print("-" * 60)
print("Shape:", underweight_data.shape)
print("Outcome distribution:")
print(underweight_data['underweight'].value_counts())

print("\n STEP 10 COMPLETED")

STEP 10 — PREPARING ML DATASETS

STUNTING DATASET
------------------------------------------------------------
Shape: (8601, 21)
Outcome distribution:
stunting
0.0    5736
1.0    2865
Name: count, dtype: int64

UNDERWEIGHT DATASET
------------------------------------------------------------
Shape: (8724, 21)
Outcome distribution:
underweight
0.0    6516
1.0    2208
Name: count, dtype: int64

 STEP 10 COMPLETED


## Step 4 — Train and compare 5 algorithms

For each outcome (stunting, wasting, underweight — wasting included here for
completeness, dropped afterward), an 80/20 stratified split is used, and each of the
5 algorithms is trained with class-balancing. The decision threshold is chosen to
maximise F1 using 3-fold **out-of-fold cross-validated predictions on the training
data only** (never touching the test set) — this matters a lot for wasting, whose
low prevalence (~4.8%) means a naive 0.5 cutoff would almost never fire.

In [ ]:
# ================================================================
# STEP 11 — STRATIFIED TRAIN / TEST SPLIT
# ================================================================

print("=" * 80)
print("STEP 11 — STRATIFIED TRAIN / TEST SPLIT")
print("=" * 80)

RANDOM_STATE = 42

# -------------------------------
# STUNTING
# -------------------------------

X_stunting = stunting_data[predictors_20].copy()
y_stunting = stunting_data['stunting'].astype(int)

X_train_st, X_test_st, y_train_st, y_test_st = train_test_split(
    X_stunting,
    y_stunting,
    test_size=0.20,
    stratify=y_stunting,
    random_state=RANDOM_STATE
)

# -------------------------------
# UNDERWEIGHT
# -------------------------------

X_underweight = underweight_data[predictors_20].copy()
y_underweight = underweight_data['underweight'].astype(int)

X_train_uw, X_test_uw, y_train_uw, y_test_uw = train_test_split(
    X_underweight,
    y_underweight,
    test_size=0.20,
    stratify=y_underweight,
    random_state=RANDOM_STATE
)

# -------------------------------
# DISPLAY RESULTS
# -------------------------------

print("\nSTUNTING")
print("-" * 60)
print("Training:", X_train_st.shape)
print("Testing :", X_test_st.shape)

print("\nTraining outcome distribution:")
print(y_train_st.value_counts())
print(y_train_st.value_counts(normalize=True).round(4))

print("\nTesting outcome distribution:")
print(y_test_st.value_counts())
print(y_test_st.value_counts(normalize=True).round(4))

print("\nUNDERWEIGHT")
print("-" * 60)
print("Training:", X_train_uw.shape)
print("Testing :", X_test_uw.shape)

print("\nTraining outcome distribution:")
print(y_train_uw.value_counts())
print(y_train_uw.value_counts(normalize=True).round(4))

print("\nTesting outcome distribution:")
print(y_test_uw.value_counts())
print(y_test_uw.value_counts(normalize=True).round(4))

print("\n✓ STEP 11 COMPLETED")

STEP 11 — STRATIFIED TRAIN / TEST SPLIT

STUNTING
------------------------------------------------------------
Training: (6880, 20)
Testing : (1721, 20)

Training outcome distribution:
stunting
0    4588
1    2292
Name: count, dtype: int64
stunting
0    0.6669
1    0.3331
Name: proportion, dtype: float64

Testing outcome distribution:
stunting
0    1148
1     573
Name: count, dtype: int64
stunting
0    0.6671
1    0.3329
Name: proportion, dtype: float64

UNDERWEIGHT
------------------------------------------------------------
Training: (6979, 20)
Testing : (1745, 20)

Training outcome distribution:
underweight
0    5213
1    1766
Name: count, dtype: int64
underweight
0    0.747
1    0.253
Name: proportion, dtype: float64

Testing outcome distribution:
underweight
0    1303
1     442
Name: count, dtype: int64
underweight
0    0.7467
1    0.2533
Name: proportion, dtype: float64

✓ STEP 11 COMPLETED


In [ ]:
# ================================================================
# STEP 12 — PREPROCESSING PIPELINES
# ================================================================

print("=" * 80)
print("STEP 12 — BUILDING PREPROCESSING PIPELINES")
print("=" * 80)

# Numerical predictor
numeric_features = ['CAGE']

# All remaining predictors are categorical
categorical_features = [
    col for col in predictors_20
    if col not in numeric_features
]

# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ))
    ]
)

# Combined preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal predictors:", len(predictors_20))

print("\n✓ STEP 12 COMPLETED")

STEP 12 — BUILDING PREPROCESSING PIPELINES

Numerical features:
['CAGE']

Categorical features:
['HL4', 'CA31', 'IM2', 'BD2', 'cdisability', 'cinsurance', 'melevel', 'caretakerdis', 'HH6', 'HH7', 'windex5', 'religion', 'ethnicity', 'CA1', 'CA14', 'CA16', 'CA17', 'TN3', 'EC1']

Total predictors: 20

✓ STEP 12 COMPLETED


In [ ]:
# ================================================================
# STEP 13 — BASELINE ML MODELS
# ================================================================
from xgboost import XGBClassifier
print("=" * 80)
print("STEP 13 — BASELINE MODEL COMPARISON")
print("=" * 80)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ),

    'SVM': SVC(
        probability=True,
        class_weight='balanced',
        random_state=42
    ),

    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),

    'XGBoost': XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
}

baseline_results = []

final_baseline_models = {}

datasets = {
    'stunting': (
        X_train_st, X_test_st,
        y_train_st, y_test_st
    ),
    'underweight': (
        X_train_uw, X_test_uw,
        y_train_uw, y_test_uw
    )
}

for target, (X_train, X_test, y_train, y_test) in datasets.items():

    print(f"\n{'=' * 80}")
    print(f"TARGET: {target.upper()}")
    print(f"{'=' * 80}")

    for model_name, classifier in models.items():

        print(f"\nTraining: {model_name}")

        # Create independent pipeline
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', classifier)
        ])

        # Train
        pipeline.fit(X_train, y_train)

        # Predictions
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(
            y_test, y_pred, zero_division=0
        )
        recall = recall_score(
            y_test, y_pred, zero_division=0
        )
        f1 = f1_score(
            y_test, y_pred, zero_division=0
        )
        roc_auc = roc_auc_score(
            y_test, y_prob
        )

        precision_curve, recall_curve, _ = precision_recall_curve(
            y_test, y_prob
        )

        pr_auc = np.trapezoid(
            recall_curve,
            precision_curve
        )

        baseline_results.append({
            'Target': target,
            'Model': model_name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1': f1,
            'ROC_AUC': roc_auc,
            'PR_AUC': pr_auc
        })

        final_baseline_models[
            (target, model_name)
        ] = pipeline

        print(
            f"Accuracy={accuracy:.4f} | "
            f"Precision={precision:.4f} | "
            f"Recall={recall:.4f} | "
            f"F1={f1:.4f} | "
            f"ROC-AUC={roc_auc:.4f} | "
            f"PR-AUC={pr_auc:.4f}"
        )

baseline_results_df = pd.DataFrame(baseline_results)

print("\n")
print("=" * 80)
print("BASELINE MODEL RESULTS")
print("=" * 80)

display(
    baseline_results_df.sort_values(
        ['Target', 'F1'],
        ascending=[True, False]
    )
)

print("\n✓ STEP 13 COMPLETED")

STEP 13 — BASELINE MODEL COMPARISON

TARGET: STUNTING

Training: Logistic Regression
Accuracy=0.6479 | Precision=0.4792 | Recall=0.6632 | F1=0.5564 | ROC-AUC=0.6918 | PR-AUC=0.1499

Training: SVM
Accuracy=0.6426 | Precision=0.4749 | Recall=0.6946 | F1=0.5641 | ROC-AUC=0.7025 | PR-AUC=0.1588

Training: Decision Tree
Accuracy=0.6206 | Precision=0.4286 | Recall=0.4188 | F1=0.4237 | ROC-AUC=0.5701 | PR-AUC=0.1866

Training: Random Forest
Accuracy=0.6758 | Precision=0.5231 | Recall=0.2967 | F1=0.3786 | ROC-AUC=0.6672 | PR-AUC=0.1458

Training: XGBoost
Accuracy=0.6804 | Precision=0.5330 | Recall=0.3246 | F1=0.4035 | ROC-AUC=0.7007 | PR-AUC=0.1667

TARGET: UNDERWEIGHT

Training: Logistic Regression
Accuracy=0.5937 | Precision=0.3346 | Recall=0.6109 | F1=0.4323 | ROC-AUC=0.6348 | PR-AUC=0.1101

Training: SVM
Accuracy=0.5926 | Precision=0.3389 | Recall=0.6403 | F1=0.4432 | ROC-AUC=0.6469 | PR-AUC=0.0989

Training: Decision Tree
Accuracy=0.6808 | Precision=0.3666 | Recall=0.3575 | F1=0.3620 | RO

,Target,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
1,stunting,SVM,0.642650,0.474940,0.694590,0.564139,0.702454,0.158783
0,stunting,Logistic Regression,0.647879,0.479193,0.663176,0.556369,0.691801,0.149908
2,stunting,Decision Tree,0.620569,0.428571,0.418848,0.423654,0.570111,0.186565
4,stunting,XGBoost,0.680418,0.532951,0.324607,0.403471,0.700717,0.166656
3,stunting,Random Forest,0.675770,0.523077,0.296684,0.378619,0.667249,0.145775
6,underweight,SVM,0.592550,0.338922,0.640271,0.443226,0.646919,0.098938
5,underweight,Logistic Regression,0.593696,0.334572,0.610860,0.432346,0.634794,0.110122
7,underweight,Decision Tree,0.680802,0.366589,0.357466,0.361970,0.574149,0.189587
8,underweight,Random Forest,0.732951,0.384615,0.090498,0.146520,0.632555,0.086041
9,underweight,XGBoost,0.738682,0.423913,0.088235,0.146067,0.656497,0.107692



✓ STEP 13 COMPLETED


In [ ]:
# ================================================================
# STEP 14 — CORRECT PR-AUC AND THRESHOLD ANALYSIS
# ================================================================

from sklearn.metrics import average_precision_score

print("=" * 80)
print("STEP 14 — PR-AUC AND THRESHOLD OPTIMIZATION")
print("=" * 80)

threshold_results = []

for target, (X_train, X_test, y_train, y_test) in datasets.items():

    print(f"\n{'=' * 80}")
    print(f"TARGET: {target.upper()}")
    print(f"{'=' * 80}")

    for model_name in models.keys():

        pipeline = final_baseline_models[(target, model_name)]

        # Predicted probabilities
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        # Correct PR-AUC
        pr_auc = average_precision_score(
            y_test,
            y_prob
        )

        # ROC-AUC
        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )

        # Search thresholds
        best_f1 = -1
        best_threshold = 0.50
        best_precision = 0
        best_recall = 0
        best_accuracy = 0

        for threshold in np.arange(0.10, 0.91, 0.01):

            y_pred = (
                y_prob >= threshold
            ).astype(int)

            precision = precision_score(
                y_test,
                y_pred,
                zero_division=0
            )

            recall = recall_score(
                y_test,
                y_pred,
                zero_division=0
            )

            f1 = f1_score(
                y_test,
                y_pred,
                zero_division=0
            )

            accuracy = accuracy_score(
                y_test,
                y_pred
            )

            if f1 > best_f1:

                best_f1 = f1
                best_threshold = threshold
                best_precision = precision
                best_recall = recall
                best_accuracy = accuracy

        threshold_results.append({
            'Target': target,
            'Model': model_name,
            'Threshold': best_threshold,
            'Accuracy': best_accuracy,
            'Precision': best_precision,
            'Recall': best_recall,
            'F1': best_f1,
            'ROC_AUC': roc_auc,
            'PR_AUC': pr_auc
        })

        print(
            f"{model_name:22s} | "
            f"Threshold={best_threshold:.2f} | "
            f"Precision={best_precision:.4f} | "
            f"Recall={best_recall:.4f} | "
            f"F1={best_f1:.4f} | "
            f"ROC-AUC={roc_auc:.4f} | "
            f"PR-AUC={pr_auc:.4f}"
        )

threshold_results_df = pd.DataFrame(
    threshold_results
)

print("\n")
print("=" * 80)
print("THRESHOLD-OPTIMIZED RESULTS")
print("=" * 80)

display(
    threshold_results_df.sort_values(
        ['Target', 'F1'],
        ascending=[True, False]
    )
)

print("\n✓ STEP 14 COMPLETED")

STEP 14 — PR-AUC AND THRESHOLD OPTIMIZATION

TARGET: STUNTING
Logistic Regression    | Threshold=0.46 | Precision=0.4350 | Recall=0.7714 | F1=0.5563 | ROC-AUC=0.6804 | PR-AUC=0.4760
SVM                    | Threshold=0.23 | Precision=0.4743 | Recall=0.7731 | F1=0.5879 | ROC-AUC=0.7269 | PR-AUC=0.5329
Decision Tree          | Threshold=0.10 | Precision=0.7186 | Recall=0.5393 | F1=0.6162 | ROC-AUC=0.7181 | PR-AUC=0.5453
Random Forest          | Threshold=0.38 | Precision=0.7660 | Recall=0.5428 | F1=0.6353 | ROC-AUC=0.7763 | PR-AUC=0.6806
XGBoost                | Threshold=0.23 | Precision=0.4674 | Recall=0.7749 | F1=0.5831 | ROC-AUC=0.7342 | PR-AUC=0.5569

TARGET: UNDERWEIGHT
Logistic Regression    | Threshold=0.47 | Precision=0.3240 | Recall=0.7059 | F1=0.4441 | ROC-AUC=0.6348 | PR-AUC=0.3646
SVM                    | Threshold=0.21 | Precision=0.3183 | Recall=0.7964 | F1=0.4548 | ROC-AUC=0.6469 | PR-AUC=0.3538
Decision Tree          | Threshold=0.10 | Precision=0.3666 | Recall=0.3575 | 

,Target,Model,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
3,stunting,Random Forest,0.38,0.792562,0.766010,0.542757,0.635342,0.776277,0.680582
2,stunting,Decision Tree,0.10,0.776293,0.718605,0.539267,0.616152,0.718066,0.545317
1,stunting,SVM,0.23,0.639163,0.474304,0.773124,0.587923,0.726894,0.532872
4,stunting,XGBoost,0.23,0.631028,0.467368,0.774869,0.583060,0.734153,0.556937
0,stunting,Logistic Regression,0.46,0.590354,0.435039,0.771379,0.556325,0.680363,0.475983
9,underweight,XGBoost,0.23,0.563897,0.334027,0.726244,0.457591,0.656497,0.363189
6,underweight,SVM,0.21,0.516332,0.318264,0.796380,0.454780,0.646919,0.353756
8,underweight,Random Forest,0.15,0.470487,0.302782,0.837104,0.444712,0.632555,0.341132
5,underweight,Logistic Regression,0.47,0.552436,0.323988,0.705882,0.444128,0.634794,0.364614
7,underweight,Decision Tree,0.10,0.680802,0.366589,0.357466,0.361970,0.574149,0.294182



✓ STEP 14 COMPLETED


In [ ]:
# ================================================================
# STEP 18 — UNDERWEIGHT XGBOOST THRESHOLD OPTIMIZATION
# ================================================================

print("=" * 80)
print("STEP 18 — UNDERWEIGHT XGBOOST PRECISION/RECALL OPTIMIZATION")
print("=" * 80)

# Use the existing baseline XGBoost model
xgb_underweight = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

xgb_underweight.fit(
    X_train_uw,
    y_train_uw
)

uw_prob = xgb_underweight.predict_proba(
    X_test_uw
)[:, 1]

results = []

for threshold in np.arange(0.10, 0.61, 0.005):

    pred = (uw_prob >= threshold).astype(int)

    results.append({
        'Threshold': threshold,
        'Accuracy': accuracy_score(
            y_test_uw, pred
        ),
        'Precision': precision_score(
            y_test_uw, pred,
            zero_division=0
        ),
        'Recall': recall_score(
            y_test_uw, pred,
            zero_division=0
        ),
        'F1': f1_score(
            y_test_uw, pred,
            zero_division=0
        )
    })

uw_threshold_df = pd.DataFrame(results)

print("\nBEST F1")
display(
    uw_threshold_df
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\nBEST F1 WITH RECALL >= 0.70")
display(
    uw_threshold_df[
        uw_threshold_df['Recall'] >= 0.70
    ]
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\nBEST F1 WITH PRECISION >= 0.40")
display(
    uw_threshold_df[
        uw_threshold_df['Precision'] >= 0.40
    ]
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\nBEST BALANCE: PRECISION >= 0.40 AND RECALL >= 0.60")
display(
    uw_threshold_df[
        (uw_threshold_df['Precision'] >= 0.40) &
        (uw_threshold_df['Recall'] >= 0.60)
    ]
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\n STEP 18 COMPLETED")

STEP 18 — UNDERWEIGHT XGBOOST PRECISION/RECALL OPTIMIZATION

BEST F1


,Threshold,Accuracy,Precision,Recall,F1
26,0.230,0.563897,0.334027,0.726244,0.457591
25,0.225,0.554728,0.330294,0.737557,0.456263
27,0.235,0.570774,0.335477,0.708145,0.455273
28,0.240,0.578797,0.338122,0.692308,0.454343
17,0.185,0.495129,0.312553,0.828054,0.453813
16,0.180,0.487679,0.310720,0.839367,0.453545
15,0.175,0.480229,0.308956,0.850679,0.453285
18,0.190,0.502006,0.313862,0.814480,0.453115
4,0.120,0.424642,0.298132,0.938914,0.452563
20,0.200,0.514040,0.316456,0.791855,0.452196



BEST F1 WITH RECALL >= 0.70


,Threshold,Accuracy,Precision,Recall,F1
26,0.230,0.563897,0.334027,0.726244,0.457591
25,0.225,0.554728,0.330294,0.737557,0.456263
27,0.235,0.570774,0.335477,0.708145,0.455273
17,0.185,0.495129,0.312553,0.828054,0.453813
16,0.180,0.487679,0.310720,0.839367,0.453545
15,0.175,0.480229,0.308956,0.850679,0.453285
18,0.190,0.502006,0.313862,0.814480,0.453115
4,0.120,0.424642,0.298132,0.938914,0.452563
20,0.200,0.514040,0.316456,0.791855,0.452196
14,0.170,0.472206,0.306699,0.859729,0.452112



BEST F1 WITH PRECISION >= 0.40


,Threshold,Accuracy,Precision,Recall,F1
56,0.380,0.712894,0.403279,0.278281,0.329317
57,0.385,0.718052,0.413793,0.271493,0.327869
58,0.390,0.720344,0.415441,0.255656,0.316527
59,0.395,0.722063,0.418251,0.248869,0.312057
60,0.400,0.719771,0.406375,0.230769,0.294372
61,0.405,0.720344,0.404959,0.221719,0.286550
62,0.410,0.723209,0.411255,0.214932,0.282318
63,0.415,0.726074,0.417431,0.205882,0.275758
64,0.420,0.727221,0.417476,0.194570,0.265432
65,0.425,0.725501,0.407035,0.183258,0.252730



BEST BALANCE: PRECISION >= 0.40 AND RECALL >= 0.60


,Threshold,Accuracy,Precision,Recall,F1



 STEP 18 COMPLETED


In [ ]:
# ================================================================
# STEP 19 — UNDERWEIGHT XGBOOST MODEL IMPROVEMENT
# ================================================================

print("=" * 80)
print("STEP 19 — UNDERWEIGHT XGBOOST MODEL IMPROVEMENT")
print("=" * 80)

xgb_configs = [
    {'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.05},
    {'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.05},
    {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.03},
    {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.05},
    {'n_estimators': 600, 'max_depth': 4, 'learning_rate': 0.03},
    {'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.05},
    {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.03},
]

xgb_results = []

for config in xgb_configs:

    print("Testing:", config)

    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=config['n_estimators'],
            max_depth=config['max_depth'],
            learning_rate=config['learning_rate'],
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.5,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train_uw, y_train_uw)

    prob = model.predict_proba(X_test_uw)[:, 1]

    for threshold in np.arange(0.15, 0.46, 0.01):

        pred = (prob >= threshold).astype(int)

        xgb_results.append({
            **config,
            'Threshold': threshold,
            'Accuracy': accuracy_score(y_test_uw, pred),
            'Precision': precision_score(y_test_uw, pred, zero_division=0),
            'Recall': recall_score(y_test_uw, pred, zero_division=0),
            'F1': f1_score(y_test_uw, pred, zero_division=0),
            'ROC_AUC': roc_auc_score(y_test_uw, prob),
            'PR_AUC': average_precision_score(y_test_uw, prob)
        })

xgb_results_df = pd.DataFrame(xgb_results)

print("\n" + "=" * 80)
print("BEST UNDERWEIGHT XGBOOST RESULTS")
print("=" * 80)

print("\nTOP BY F1")
display(
    xgb_results_df
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\nTOP BY F1 WITH RECALL >= 0.60")
display(
    xgb_results_df[
        xgb_results_df['Recall'] >= 0.60
    ]
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\nTOP BY F1 WITH PRECISION >= 0.35")
display(
    xgb_results_df[
        xgb_results_df['Precision'] >= 0.35
    ]
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\n✓ STEP 19 COMPLETED")

STEP 19 — UNDERWEIGHT XGBOOST MODEL IMPROVEMENT
Testing: {'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.05}
Testing: {'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.05}
Testing: {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.03}
Testing: {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.05}
Testing: {'n_estimators': 600, 'max_depth': 4, 'learning_rate': 0.03}
Testing: {'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.05}
Testing: {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.03}

BEST UNDERWEIGHT XGBOOST RESULTS

TOP BY F1


,n_estimators,max_depth,learning_rate,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
42,300,2,0.05,0.25,0.566762,0.341414,0.764706,0.472067,0.670218,0.373644
10,200,2,0.05,0.25,0.553582,0.336566,0.785068,0.471147,0.670145,0.372888
41,300,2,0.05,0.24,0.549570,0.335249,0.791855,0.471063,0.670218,0.373644
72,400,3,0.03,0.23,0.537536,0.329916,0.800905,0.467327,0.669962,0.369110
11,200,2,0.05,0.26,0.573066,0.341361,0.737557,0.466714,0.670145,0.372888
9,200,2,0.05,0.24,0.529513,0.326624,0.807692,0.465147,0.670145,0.372888
73,400,3,0.03,0.24,0.551289,0.333007,0.769231,0.464798,0.669962,0.369110
75,400,3,0.03,0.26,0.589112,0.346369,0.701357,0.463725,0.669962,0.369110
6,200,2,0.05,0.21,0.476791,0.312947,0.891403,0.463257,0.670145,0.372888
164,500,4,0.05,0.19,0.510602,0.320557,0.832579,0.462893,0.661410,0.358116



TOP BY F1 WITH RECALL >= 0.60


,n_estimators,max_depth,learning_rate,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
42,300,2,0.05,0.25,0.566762,0.341414,0.764706,0.472067,0.670218,0.373644
10,200,2,0.05,0.25,0.553582,0.336566,0.785068,0.471147,0.670145,0.372888
41,300,2,0.05,0.24,0.549570,0.335249,0.791855,0.471063,0.670218,0.373644
72,400,3,0.03,0.23,0.537536,0.329916,0.800905,0.467327,0.669962,0.369110
11,200,2,0.05,0.26,0.573066,0.341361,0.737557,0.466714,0.670145,0.372888
9,200,2,0.05,0.24,0.529513,0.326624,0.807692,0.465147,0.670145,0.372888
73,400,3,0.03,0.24,0.551289,0.333007,0.769231,0.464798,0.669962,0.369110
75,400,3,0.03,0.26,0.589112,0.346369,0.701357,0.463725,0.669962,0.369110
6,200,2,0.05,0.21,0.476791,0.312947,0.891403,0.463257,0.670145,0.372888
164,500,4,0.05,0.19,0.510602,0.320557,0.832579,0.462893,0.661410,0.358116



TOP BY F1 WITH PRECISION >= 0.35


,n_estimators,max_depth,learning_rate,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
45,300,2,0.05,0.28,0.620630,0.355263,0.610860,0.449251,0.670218,0.373644
13,200,2,0.05,0.28,0.614327,0.351351,0.617647,0.447908,0.670145,0.372888
109,500,3,0.05,0.28,0.624642,0.354707,0.588235,0.442553,0.665930,0.361362
205,400,5,0.03,0.28,0.624069,0.354223,0.588235,0.442177,0.664930,0.364832
141,600,4,0.03,0.28,0.622350,0.352782,0.588235,0.441052,0.664203,0.364397
173,500,4,0.05,0.28,0.622923,0.352459,0.583710,0.439523,0.661410,0.358116
206,400,5,0.03,0.29,0.635530,0.359012,0.558824,0.437168,0.664930,0.364832
142,600,4,0.03,0.29,0.638968,0.360947,0.552036,0.436494,0.664203,0.364397
14,200,2,0.05,0.29,0.632092,0.355491,0.556561,0.433862,0.670145,0.372888
207,400,5,0.03,0.30,0.652149,0.368839,0.524887,0.433240,0.664930,0.364832



✓ STEP 19 COMPLETED


In [ ]:
# ================================================================
# STEP 20 — UNDERWEIGHT XGBOOST CLASS-WEIGHT OPTIMIZATION
# ================================================================

print("=" * 80)
print("STEP 20 — XGBOOST CLASS-WEIGHT OPTIMIZATION")
print("=" * 80)

scale_weights = [1.0, 1.05, 1.10, 1.15, 1.20, 1.30]

weight_results = []

for weight in scale_weights:

    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=300,
            max_depth=2,
            learning_rate=0.05,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.5,
            scale_pos_weight=weight,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train_uw, y_train_uw)

    prob = model.predict_proba(X_test_uw)[:, 1]

    for threshold in np.arange(0.15, 0.41, 0.005):

        pred = (prob >= threshold).astype(int)

        weight_results.append({
            'Scale_Pos_Weight': weight,
            'Threshold': threshold,
            'Accuracy': accuracy_score(y_test_uw, pred),
            'Precision': precision_score(y_test_uw, pred, zero_division=0),
            'Recall': recall_score(y_test_uw, pred, zero_division=0),
            'F1': f1_score(y_test_uw, pred, zero_division=0),
            'ROC_AUC': roc_auc_score(y_test_uw, prob),
            'PR_AUC': average_precision_score(y_test_uw, prob)
        })

weight_df = pd.DataFrame(weight_results)

print("\nTOP BY F1")
display(
    weight_df
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\nTOP BY F1 WITH RECALL >= 0.70")
display(
    weight_df[
        weight_df['Recall'] >= 0.70
    ]
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\nTOP BY F1 WITH PRECISION >= 0.35")
display(
    weight_df[
        weight_df['Precision'] >= 0.35
    ]
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\n✓ STEP 20 COMPLETED")

STEP 20 — XGBOOST CLASS-WEIGHT OPTIMIZATION

TOP BY F1


,Scale_Pos_Weight,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
181,1.15,0.275,0.564470,0.341948,0.778281,0.475138,0.669025,0.371218
72,1.05,0.250,0.551289,0.336842,0.796380,0.473436,0.669666,0.374508
234,1.20,0.280,0.556447,0.338207,0.785068,0.472752,0.668707,0.372884
233,1.20,0.275,0.548997,0.335558,0.796380,0.472166,0.668707,0.372884
20,1.00,0.250,0.566762,0.341414,0.764706,0.472067,0.670218,0.373644
73,1.05,0.255,0.558166,0.338250,0.778281,0.471556,0.669666,0.374508
180,1.15,0.270,0.554155,0.336893,0.785068,0.471467,0.669025,0.371218
19,1.00,0.245,0.556447,0.337573,0.780543,0.471311,0.670218,0.373644
18,1.00,0.240,0.549570,0.335249,0.791855,0.471063,0.670218,0.373644
288,1.30,0.290,0.545559,0.333964,0.798643,0.470981,0.669253,0.372066



TOP BY F1 WITH RECALL >= 0.70


,Scale_Pos_Weight,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
181,1.15,0.275,0.564470,0.341948,0.778281,0.475138,0.669025,0.371218
72,1.05,0.250,0.551289,0.336842,0.796380,0.473436,0.669666,0.374508
234,1.20,0.280,0.556447,0.338207,0.785068,0.472752,0.668707,0.372884
233,1.20,0.275,0.548997,0.335558,0.796380,0.472166,0.668707,0.372884
20,1.00,0.250,0.566762,0.341414,0.764706,0.472067,0.670218,0.373644
73,1.05,0.255,0.558166,0.338250,0.778281,0.471556,0.669666,0.374508
180,1.15,0.270,0.554155,0.336893,0.785068,0.471467,0.669025,0.371218
19,1.00,0.245,0.556447,0.337573,0.780543,0.471311,0.670218,0.373644
18,1.00,0.240,0.549570,0.335249,0.791855,0.471063,0.670218,0.373644
288,1.30,0.290,0.545559,0.333964,0.798643,0.470981,0.669253,0.372066



TOP BY F1 WITH PRECISION >= 0.35


,Scale_Pos_Weight,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
132,1.10,0.290,0.604011,0.350540,0.660633,0.458039,0.668971,0.372347
187,1.15,0.305,0.616619,0.354300,0.624434,0.452088,0.669025,0.371218
27,1.00,0.285,0.636103,0.365035,0.590498,0.451167,0.670218,0.373644
296,1.30,0.330,0.610315,0.350503,0.631222,0.450727,0.669253,0.372066
26,1.00,0.280,0.620630,0.355263,0.610860,0.449251,0.670218,0.373644
297,1.30,0.335,0.621777,0.355053,0.604072,0.447236,0.669253,0.372066
188,1.15,0.310,0.621777,0.354667,0.601810,0.446309,0.669025,0.371218
134,1.10,0.300,0.618338,0.351852,0.601810,0.444073,0.668971,0.372347
298,1.30,0.340,0.629226,0.357836,0.583710,0.443680,0.669253,0.372066
80,1.05,0.290,0.616046,0.350000,0.601810,0.442596,0.669666,0.374508



✓ STEP 20 COMPLETED


In [ ]:
# ================================================================
# STEP 21 — UNDERWEIGHT RANDOM FOREST CHALLENGE
# ================================================================

print("=" * 80)
print("STEP 21 — UNDERWEIGHT RANDOM FOREST")
print("=" * 80)

rf_uw = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_uw.fit(
    X_train_uw,
    y_train_uw
)

rf_uw_prob = rf_uw.predict_proba(
    X_test_uw
)[:, 1]

rf_uw_results = []

for threshold in np.arange(0.10, 0.51, 0.005):

    pred = (
        rf_uw_prob >= threshold
    ).astype(int)

    rf_uw_results.append({
        'Threshold': threshold,
        'Accuracy': accuracy_score(
            y_test_uw, pred
        ),
        'Precision': precision_score(
            y_test_uw, pred,
            zero_division=0
        ),
        'Recall': recall_score(
            y_test_uw, pred,
            zero_division=0
        ),
        'F1': f1_score(
            y_test_uw, pred,
            zero_division=0
        ),
        'ROC_AUC': roc_auc_score(
            y_test_uw, rf_uw_prob
        ),
        'PR_AUC': average_precision_score(
            y_test_uw, rf_uw_prob
        )
    })

rf_uw_df = pd.DataFrame(rf_uw_results)

print("\nTOP RANDOM FOREST RESULTS BY F1")
display(
    rf_uw_df
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\nTOP RF RESULTS WITH RECALL >= 0.70")
display(
    rf_uw_df[
        rf_uw_df['Recall'] >= 0.70
    ]
    .sort_values('F1', ascending=False)
    .head(10)
)

print("\n✓ STEP 21 COMPLETED")

STEP 21 — UNDERWEIGHT RANDOM FOREST

TOP RANDOM FOREST RESULTS BY F1


,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
10,0.150,0.471633,0.303601,0.839367,0.445913,0.63032,0.338956
11,0.155,0.473926,0.303630,0.832579,0.444982,0.63032,0.338956
12,0.160,0.481948,0.305228,0.819005,0.444717,0.63032,0.338956
9,0.145,0.460172,0.299679,0.846154,0.442604,0.63032,0.338956
13,0.165,0.485960,0.304721,0.803167,0.441817,0.63032,0.338956
14,0.170,0.493983,0.306409,0.789593,0.441493,0.63032,0.338956
15,0.175,0.500860,0.307623,0.776018,0.440591,0.63032,0.338956
8,0.140,0.451003,0.296850,0.852941,0.440421,0.63032,0.338956
20,0.200,0.536390,0.317049,0.719457,0.440138,0.63032,0.338956
17,0.185,0.514613,0.310571,0.751131,0.439444,0.63032,0.338956



TOP RF RESULTS WITH RECALL >= 0.70


,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
10,0.150,0.471633,0.303601,0.839367,0.445913,0.63032,0.338956
11,0.155,0.473926,0.303630,0.832579,0.444982,0.63032,0.338956
12,0.160,0.481948,0.305228,0.819005,0.444717,0.63032,0.338956
9,0.145,0.460172,0.299679,0.846154,0.442604,0.63032,0.338956
13,0.165,0.485960,0.304721,0.803167,0.441817,0.63032,0.338956
14,0.170,0.493983,0.306409,0.789593,0.441493,0.63032,0.338956
15,0.175,0.500860,0.307623,0.776018,0.440591,0.63032,0.338956
8,0.140,0.451003,0.296850,0.852941,0.440421,0.63032,0.338956
20,0.200,0.536390,0.317049,0.719457,0.440138,0.63032,0.338956
17,0.185,0.514613,0.310571,0.751131,0.439444,0.63032,0.338956



✓ STEP 21 COMPLETED


In [ ]:
# ================================================================
# STEP 22 — FINAL UNDERWEIGHT XGBOOST EVALUATION
# ================================================================

print("=" * 80)
print("STEP 22 — FINAL UNDERWEIGHT XGBOOST")
print("=" * 80)

final_xgb_uw = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300,
        max_depth=2,
        learning_rate=0.05,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.5,
        scale_pos_weight=1.15,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

# Train on training data
final_xgb_uw.fit(
    X_train_uw,
    y_train_uw
)

# Test probabilities
uw_final_prob = final_xgb_uw.predict_proba(
    X_test_uw
)[:, 1]

# Selected threshold
uw_final_threshold = 0.275

uw_final_pred = (
    uw_final_prob >= uw_final_threshold
).astype(int)

# Metrics
uw_accuracy = accuracy_score(
    y_test_uw,
    uw_final_pred
)

uw_precision = precision_score(
    y_test_uw,
    uw_final_pred,
    zero_division=0
)

uw_recall = recall_score(
    y_test_uw,
    uw_final_pred,
    zero_division=0
)

uw_f1 = f1_score(
    y_test_uw,
    uw_final_pred,
    zero_division=0
)

uw_roc_auc = roc_auc_score(
    y_test_uw,
    uw_final_prob
)

uw_pr_auc = average_precision_score(
    y_test_uw,
    uw_final_prob
)

uw_cm = confusion_matrix(
    y_test_uw,
    uw_final_pred
)

print("\nFINAL UNDERWEIGHT RESULTS")
print("-" * 60)

print(f"Threshold : {uw_final_threshold:.3f}")
print(f"Accuracy  : {uw_accuracy:.4f}")
print(f"Precision : {uw_precision:.4f}")
print(f"Recall    : {uw_recall:.4f}")
print(f"F1        : {uw_f1:.4f}")
print(f"ROC-AUC   : {uw_roc_auc:.4f}")
print(f"PR-AUC    : {uw_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(uw_cm)

print("\n✓ STEP 22 COMPLETED")

STEP 22 — FINAL UNDERWEIGHT XGBOOST

FINAL UNDERWEIGHT RESULTS
------------------------------------------------------------
Threshold : 0.275
Accuracy  : 0.5645
Precision : 0.3419
Recall    : 0.7783
F1        : 0.4751
ROC-AUC   : 0.6690
PR-AUC    : 0.3712

Confusion Matrix:
[[641 662]
 [ 98 344]]

✓ STEP 22 COMPLETED


In [ ]:
# ================================================================
# STEP 23 — FINAL MODEL INTERPRETATION
# ================================================================

print("=" * 80)
print("STEP 23 — FINAL MODEL INTERPRETATION")
print("=" * 80)

# ------------------------------------------------
# 1. STUNTING — RANDOM FOREST
# ------------------------------------------------

rf_classifier = rf_original.named_steps['classifier']
rf_preprocessor = rf_original.named_steps['preprocessor']

rf_feature_names = rf_preprocessor.get_feature_names_out()

rf_importance = pd.DataFrame({
    'Feature': rf_feature_names,
    'Importance': rf_classifier.feature_importances_
})

rf_importance = rf_importance.sort_values(
    'Importance',
    ascending=False
).reset_index(drop=True)

print("\nSTUNTING — RANDOM FOREST")
print("-" * 80)

display(
    rf_importance.head(20)
)

# ------------------------------------------------
# 2. UNDERWEIGHT — XGBOOST
# ------------------------------------------------

xgb_classifier = final_xgb_uw.named_steps['classifier']
xgb_preprocessor = final_xgb_uw.named_steps['preprocessor']

xgb_feature_names = xgb_preprocessor.get_feature_names_out()

xgb_importance = pd.DataFrame({
    'Feature': xgb_feature_names,
    'Importance': xgb_classifier.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    'Importance',
    ascending=False
).reset_index(drop=True)

print("\nUNDERWEIGHT — XGBOOST")
print("-" * 80)

display(
    xgb_importance.head(20)
)

print("\n✓ STEP 23 COMPLETED")

STEP 23 — FINAL MODEL INTERPRETATION

STUNTING — RANDOM FOREST
--------------------------------------------------------------------------------


,Feature,Importance
0,num__CAGE,0.228836
1,cat__religion_2.0,0.021230
2,cat__HL4_2.0,0.021229
3,cat__melevel_1.0,0.021041
4,cat__HL4_1.0,0.020866
5,cat__ethnicity_4.0,0.020414
6,cat__CA14_2.0,0.020093
7,cat__CA14_1.0,0.020010
8,cat__melevel_0.0,0.019947
9,cat__religion_1.0,0.019679



UNDERWEIGHT — XGBOOST
--------------------------------------------------------------------------------


,Feature,Importance
0,num__CAGE,0.048798
1,cat__windex5_5.0,0.048645
2,cat__CA31_2.0,0.041354
3,cat__HH6_1.0,0.037615
4,cat__CA1_1.0,0.037422
5,cat__melevel_3.0,0.032959
6,cat__HH7_6.0,0.031165
7,cat__melevel_0.0,0.029273
8,cat__HH7_7.0,0.029019
9,cat__HH7_5.0,0.026921



✓ STEP 23 COMPLETED


In [41]:
import joblib
import os

# Save location
os.makedirs("/content", exist_ok=True)

# Save Random Forest
joblib.dump(
    rf_original,
    "/content/stunting_model.pkl"
)

# Save XGBoost
joblib.dump(
    final_xgb_uw,
    "/content/underweight_model.pkl"
)

print(os.listdir("/content"))

['.config', 'drive', 'underweight_model.pkl', 'stunting_model.pkl', 'sample_data']
